In [1]:
import pandas as pd
import numpy as np

# If df_base already exists in the notebook, use it directly.
# Otherwise, read the file.
df_src = pd.read_csv("df_base.csv", low_memory=False)

target_cols = [
    "jv.default_Jsc",
    "jv.default_Voc",
    "jv.default_FF",
    "jv.default_PCE",
]

source_cols = [
    "ref.ID",
    "ref.DOI_number",
    "ref.lead_author",
    "ref.journal",
    "ref.publication_date",
    "ref.original_filename_data_upload",
]

# Keep only rows used for complete JV/PCE modelling
for col in target_cols:
    df_src[col] = pd.to_numeric(df_src[col], errors="coerce")

df_model = df_src.dropna(subset=target_cols).copy()

print("Complete JV/PCE modelling rows:", len(df_model))

summary_rows = []

for col in source_cols:
    if col not in df_model.columns:
        continue
    
    counts = df_model[col].fillna("NA").astype(str).value_counts(dropna=False)
    
    summary_rows.append({
        "source_col": col,
        "n_groups": counts.shape[0],
        "median_group_size": counts.median(),
        "max_group_size": counts.max(),
        "n_singleton_groups": (counts == 1).sum(),
        "frac_groups_lt_5": (counts < 5).mean(),
        "rows_in_groups_lt_5": counts[counts < 5].sum(),
        "frac_rows_in_groups_lt_5": counts[counts < 5].sum() / len(df_model),
        "rows_in_groups_ge_10": counts[counts >= 10].sum(),
        "frac_rows_in_groups_ge_10": counts[counts >= 10].sum() / len(df_model),
        "80_20_stratified_possible": counts.min() >= 2,
        "10fold_stratified_possible": counts.min() >= 10,
    })

source_summary = pd.DataFrame(summary_rows)

source_summary

Complete JV/PCE modelling rows: 3953


,source_col,n_groups,median_group_size,max_group_size,n_singleton_groups,frac_groups_lt_5,rows_in_groups_lt_5,frac_rows_in_groups_lt_5,rows_in_groups_ge_10,frac_rows_in_groups_ge_10,80_20_stratified_possible,10fold_stratified_possible
0,ref.ID,3714,1.0,240,3713,0.999731,3713,0.939287,240,0.060713,False,False
1,ref.DOI_number,823,4.0,66,121,0.629405,1257,0.317986,1110,0.280799,False,False
2,ref.lead_author,421,4.0,214,65,0.553444,538,0.136099,2762,0.698710,False,False
3,ref.journal,147,8.0,389,8,0.340136,129,0.032633,3595,0.909436,False,False
4,ref.publication_date,686,4.0,66,88,0.558309,942,0.238300,1668,0.421958,False,False
5,ref.original_filename_data_upload,5,36.0,3669,1,0.200000,1,0.000253,3945,0.997976,False,False


In [2]:
doi_counts = df_model["ref.DOI_number"].fillna("NA").astype(str).value_counts(dropna=False)

doi_threshold_summary = pd.DataFrame([
    {
        "DOI_min_group_size": k,
        "n_DOI_groups": (doi_counts >= k).sum(),
        "rows_covered": doi_counts[doi_counts >= k].sum(),
        "frac_rows_covered": doi_counts[doi_counts >= k].sum() / len(df_model),
    }
    for k in [1, 2, 3, 4, 5, 10, 15, 20, 30, 50]
])

doi_threshold_summary

,DOI_min_group_size,n_DOI_groups,rows_covered,frac_rows_covered
0,1,823,3953,1.000000
1,2,702,3832,0.969390
2,3,520,3468,0.877308
3,4,432,3204,0.810524
4,5,305,2696,0.682014
5,10,70,1110,0.280799
6,15,26,608,0.153807
7,20,7,289,0.073109
8,30,5,245,0.061978
9,50,2,128,0.032380


In [3]:
df_model["ref.DOI_number"].fillna("NA").astype(str).value_counts()

ref.DOI_number
10.1039/c7ra06365b                 66
10.1021/acs.nanolett.9b00936       62
10.1021/acsaem.9b00486             43
10.1021/acsnano.7b02867            41
10.1021/acsaem.9b01298             33
                                   ..
10.1039/c5ra12530h                  1
10.1016/j.electacta.2017.07.040     1
10.1021/cm504558g                   1
10.1002/adma.201306217              1
10.1039/c4ee01624f                  1
Name: count, Length: 823, dtype: int64

## DOI-Grouped RF Sanity Check

This experiment extends the evaluation-stability analysis with a stricter
question: can the same model predict samples from DOI sources that were not
seen during training?

The canonical 207-feature all-in representation, preprocessing, Random Forest
hyperparameters, test fraction, and ten seeds are fixed to match notebook 01.
The only intended experimental change is the split strategy:

- notebook 01: random row-wise 80/20 held-out splits
- notebook 02: DOI-grouped 80/20 held-out splits

This is a diagnostic stress test of transfer to unseen literature sources, not
a model-tuning experiment and not a replacement for the main repeated random
held-out evaluation.


In [4]:
# ============================================================
# DOI-grouped fixed-RF sanity check
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

PCE_TARGET = "jv.default_PCE"
COMPONENT_TARGETS = ["jv.default_Jsc", "jv.default_Voc", "jv.default_FF"]
ALL_TARGET_COLS = COMPONENT_TARGETS + [PCE_TARGET]

DOI_GROUPED_SEEDS = list(range(10))
TEST_SIZE = 0.20
N_ESTIMATORS = 200
MIN_SAMPLES_LEAF = 1
RF_RANDOM_STATE = 42
N_JOBS_RF = -1

MAX_MISSING_RATIO = 0.50
EXPECTED_ALLIN_FEATURES = 207

PREPROCESSING_ADDED_COLUMNS = {
    "ETL_layers", "ETL_thickness_values", "ETL_n_layers",
    "ETL_n_thickness", "ETL_valid",
    "HTL_layers", "HTL_thickness_values", "HTL_n_layers",
    "HTL_n_thickness", "HTL_valid",
    "BC_layers", "BC_thickness_values", "BC_n_layers",
    "BC_n_thickness", "BC_valid",
    "ETL_thickness_clean", "HTL_thickness_clean", "BC_thickness_clean",
    "PVK_thickness_clean", "PVK_thickness_note", "PVK_valid",
    "all_layer_match", "perovskite_bandgap_clean", "baseline_complete",
}

DIRECT_OUTCOME_COLUMNS = {
    # Default, forward, and reverse JV outcomes
    "jv.default_PCE", "jv.default_Jsc", "jv.default_Voc", "jv.default_FF",
    "jv.forward_scan_PCE", "jv.forward_scan_Jsc",
    "jv.forward_scan_Voc", "jv.forward_scan_FF",
    "jv.reverse_scan_PCE", "jv.reverse_scan_Jsc",
    "jv.reverse_scan_Voc", "jv.reverse_scan_FF",

    # Other measured-curve results
    "jv.forward_scan_Vmp", "jv.forward_scan_Jmp",
    "jv.reverse_scan_Vmp", "jv.reverse_scan_Jmp",
    "jv.forward_scan_series_resistance",
    "jv.forward_scan_shunt_resistance",
    "jv.reverse_scan_series_resistance",
    "jv.reverse_scan_shunt_resistance",
    "jv.hysteresis_index",

    # Stabilised and EQE results
    "stabilised.performance_PCE", "stabilised.performance_Jsc",
    "stabilised.performance_Voc", "stabilised.performance_FF",
    "stabilised.performance_Jmp", "stabilised.performance_Vmp",
    "stabilised.performance_procedure_metrics",
    "eqe.integrated_Jsc",

    # Flags inferred from measured PCE trajectories
    "stability.PCE_burn_in_observed",
    "outdoor.PCE_burn_in_observed",
}

RAW_TO_CLEAN_REPLACEMENTS = {
    "perovskite.band_gap": "perovskite_bandgap_clean",
    "perovskite.thickness": "PVK_thickness_clean",
    "etl.thickness": "ETL_thickness_clean",
    "htl.thickness_list": "HTL_thickness_clean",
    "backcontact.thickness_list": "BC_thickness_clean",
}

LEVEL2_HELPERS = [
    "ETL_n_layers",
    "HTL_n_layers",
    "BC_n_layers",
]

LEVEL3_FEATURES = [
    "Absorber_to_ETL_ratio",
    "ff_PVK_fraction_of_total_stack",
]

NUMERIC_SOURCE_COLS = [
    "PVK_thickness_clean", "ETL_thickness_clean", "HTL_thickness_clean", "BC_thickness_clean",
    "perovskite_bandgap_clean", "jv.default_Jsc", "jv.default_Voc", "jv.default_FF", "jv.default_PCE",
    "ETL_n_layers", "HTL_n_layers", "BC_n_layers",
]


def safe_div(a, b):
    return (a / b.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)


def add_selected_level3_features(d):
    d = d.copy()
    pvk = pd.to_numeric(d["PVK_thickness_clean"], errors="coerce")
    etl = pd.to_numeric(d["ETL_thickness_clean"], errors="coerce")
    htl = pd.to_numeric(d["HTL_thickness_clean"], errors="coerce")
    bc = pd.to_numeric(d["BC_thickness_clean"], errors="coerce")

    d["Absorber_to_ETL_ratio"] = safe_div(pvk, etl)
    d["ff_PVK_fraction_of_total_stack"] = safe_div(pvk, pvk + etl + htl + bc)
    return d


def make_preprocessor(X):
    """Use the same preprocessing as notebook 01."""
    num_cols = X.select_dtypes(include=np.number).columns.tolist()
    cat_cols = [col for col in X.columns if col not in num_cols]

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
    ])
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("to_string", FunctionTransformer(lambda x: x.astype(str), validate=False)),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer([
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ])


def make_rf_pipeline(X):
    """Use the same fixed RF as notebook 01."""
    return Pipeline([
        ("prep", make_preprocessor(X)),
        ("rf", RandomForestRegressor(
            n_estimators=N_ESTIMATORS,
            min_samples_leaf=MIN_SAMPLES_LEAF,
            random_state=RF_RANDOM_STATE,
            n_jobs=N_JOBS_RF,
        )),
    ])


def count_encoded_features_after_fit(pipeline):
    prep = pipeline.named_steps["prep"]
    num_cols = prep.transformers_[0][2]
    cat_cols = prep.transformers_[1][2]
    n_num = len(num_cols)

    if len(cat_cols) == 0:
        return int(n_num)

    onehot = prep.named_transformers_["cat"].named_steps["onehot"]
    n_cat = int(sum(len(categories) for categories in onehot.categories_))
    return int(n_num + n_cat)


def rmse_score(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


# Use df_src from the audit above if available; otherwise read df_base.csv directly.
try:
    source_data = df_src.copy()
except NameError:
    source_data = pd.read_csv("df_base.csv", low_memory=False)

for col in NUMERIC_SOURCE_COLS:
    if col in source_data.columns:
        source_data[col] = pd.to_numeric(source_data[col], errors="coerce")

source_data = add_selected_level3_features(source_data).replace([np.inf, -np.inf], np.nan)

# Build canonical 207 from df_base using the same rule as notebook 01.
missing_ratio = source_data.isna().mean()

eligible_raw_features = [
    column
    for column in source_data.columns
    if missing_ratio[column] < MAX_MISSING_RATIO
    and column not in PREPROCESSING_ADDED_COLUMNS
    and column not in DIRECT_OUTCOME_COLUMNS
    and column not in ALL_TARGET_COLS
    and column not in LEVEL3_FEATURES
    and column != "m_def"
    and not column.startswith("ref.")
    and not column.endswith("link_raw_data")
]

ALLIN_FEATURES = [
    RAW_TO_CLEAN_REPLACEMENTS.get(column, column)
    for column in eligible_raw_features
]
ALLIN_FEATURES += LEVEL2_HELPERS + LEVEL3_FEATURES

required_added_features = (
    list(RAW_TO_CLEAN_REPLACEMENTS.values())
    + LEVEL2_HELPERS
    + LEVEL3_FEATURES
)
missing_added_features = [
    column for column in required_added_features
    if column not in source_data.columns
]
if missing_added_features:
    raise ValueError(
        "Missing canonical clean/helper/derived features: "
        + ", ".join(missing_added_features)
    )

# Fail loudly if the canonical definition changes.
assert len(eligible_raw_features) == 202
assert len(ALLIN_FEATURES) == EXPECTED_ALLIN_FEATURES
assert len(ALLIN_FEATURES) == len(set(ALLIN_FEATURES))
assert set(RAW_TO_CLEAN_REPLACEMENTS).isdisjoint(ALLIN_FEATURES)
assert set(RAW_TO_CLEAN_REPLACEMENTS.values()).issubset(ALLIN_FEATURES)
assert set(LEVEL2_HELPERS).issubset(ALLIN_FEATURES)
assert set(LEVEL3_FEATURES).issubset(ALLIN_FEATURES)
assert not set(ALLIN_FEATURES).intersection(DIRECT_OUTCOME_COLUMNS)
assert not any(column.startswith("ref.") for column in ALLIN_FEATURES)
assert any(column.startswith("stability.") for column in ALLIN_FEATURES)
assert any(column.startswith("outdoor.") for column in ALLIN_FEATURES)

required_cols = ALL_TARGET_COLS + ["ref.DOI_number"]
Y_all = source_data[ALL_TARGET_COLS].apply(pd.to_numeric, errors="coerce")
doi = source_data["ref.DOI_number"].astype("string").str.strip()
valid_doi = doi.notna() & (doi != "") & (doi.str.lower() != "nan")
mask = Y_all.notna().all(axis=1) & valid_doi

X_doi = (
    source_data.loc[mask, ALLIN_FEATURES]
    .copy()
    .replace({pd.NA: np.nan})
    .reset_index(drop=True)
)
y_doi = Y_all.loc[mask, PCE_TARGET].reset_index(drop=True)
groups_doi = doi.loc[mask].astype(str).reset_index(drop=True)

print("DOI-grouped sanity-check samples:", len(X_doi))
print("Unique DOI groups:", groups_doi.nunique())
print("Eligible raw descriptors:", len(eligible_raw_features))
print("Canonical all-in features:", X_doi.shape[1])
print(
    "Numeric / categorical:",
    X_doi.select_dtypes(include=np.number).shape[1],
    "/",
    X_doi.shape[1] - X_doi.select_dtypes(include=np.number).shape[1],
)


DOI-grouped sanity-check samples: 3953
Unique DOI groups: 823
Eligible raw descriptors: 202
Canonical all-in features: 207
Numeric / categorical: 20 / 187


In [5]:
# ============================================================
# Run the fixed RF on unseen DOI groups
# ============================================================

DOI_GROUPED_RF_RESULTS = []

for split_seed in DOI_GROUPED_SEEDS:
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=TEST_SIZE,
        random_state=split_seed,
    )
    train_idx, test_idx = next(
        splitter.split(X_doi, y_doi, groups=groups_doi)
    )

    X_train = X_doi.iloc[train_idx].copy()
    X_test = X_doi.iloc[test_idx].copy()
    y_train = y_doi.iloc[train_idx].copy()
    y_test = y_doi.iloc[test_idx].copy()
    groups_train = groups_doi.iloc[train_idx]
    groups_test = groups_doi.iloc[test_idx]

    # Model randomness is fixed at 42, exactly as in notebook 01.
    # Only the DOI-grouped train/test partition changes across split seeds.
    model = make_rf_pipeline(X_train)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    train_dois = set(groups_train)
    test_dois = set(groups_test)

    row = {
        "split_seed": split_seed,
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        "n_train_doi": groups_train.nunique(),
        "n_test_doi": groups_test.nunique(),
        "n_overlapping_doi": len(train_dois.intersection(test_dois)),
        "PCE_RMSE": rmse_score(y_test, y_pred),
        "PCE_R2": r2_score(y_test, y_pred),
        "n_raw_features": X_train.shape[1],
        "n_encoded_features": count_encoded_features_after_fit(model),
        "n_estimators": N_ESTIMATORS,
        "min_samples_leaf": MIN_SAMPLES_LEAF,
        "rf_random_state": RF_RANDOM_STATE,
    }
    DOI_GROUPED_RF_RESULTS.append(row)

    print(
        f"seed {split_seed:02d} | "
        f"test DOI groups={groups_test.nunique():4d} | "
        f"overlap DOI={row['n_overlapping_doi']} | "
        f"RMSE={row['PCE_RMSE']:.3f} | "
        f"R2={row['PCE_R2']:.3f}"
    )

DOI_GROUPED_RF_RESULTS = pd.DataFrame(DOI_GROUPED_RF_RESULTS)
DOI_GROUPED_RF_RESULTS


seed 00 | test DOI groups= 165 | overlap DOI=0 | RMSE=3.808 | R2=0.369
seed 01 | test DOI groups= 165 | overlap DOI=0 | RMSE=4.233 | R2=0.329
seed 02 | test DOI groups= 165 | overlap DOI=0 | RMSE=3.861 | R2=0.323
seed 03 | test DOI groups= 165 | overlap DOI=0 | RMSE=3.738 | R2=0.402
seed 04 | test DOI groups= 165 | overlap DOI=0 | RMSE=3.890 | R2=0.333
seed 05 | test DOI groups= 165 | overlap DOI=0 | RMSE=3.861 | R2=0.294
seed 06 | test DOI groups= 165 | overlap DOI=0 | RMSE=4.009 | R2=0.272
seed 07 | test DOI groups= 165 | overlap DOI=0 | RMSE=3.928 | R2=0.137
seed 08 | test DOI groups= 165 | overlap DOI=0 | RMSE=4.025 | R2=0.239
seed 09 | test DOI groups= 165 | overlap DOI=0 | RMSE=3.962 | R2=0.309


,split_seed,n_train,n_test,n_train_doi,n_test_doi,n_overlapping_doi,PCE_RMSE,PCE_R2,n_raw_features,n_encoded_features,n_estimators,min_samples_leaf,rf_random_state
0,0,3105,848,658,165,0,3.807812,0.368918,207,3815,200,1,42
1,1,3179,774,658,165,0,4.233103,0.328889,207,3844,200,1,42
2,2,3182,771,658,165,0,3.860820,0.322937,207,3852,200,1,42
3,3,3165,788,658,165,0,3.738167,0.402252,207,3750,200,1,42
4,4,3210,743,658,165,0,3.890303,0.333443,207,3849,200,1,42
5,5,3097,856,658,165,0,3.861404,0.293948,207,3735,200,1,42
6,6,3130,823,658,165,0,4.008881,0.271881,207,3859,200,1,42
7,7,3120,833,658,165,0,3.927641,0.136649,207,3819,200,1,42
8,8,3151,802,658,165,0,4.025279,0.239262,207,3885,200,1,42
9,9,3130,823,658,165,0,3.962320,0.309308,207,3837,200,1,42


In [6]:
# ============================================================
# DOI-grouped summary for appendix/report
# ============================================================

doi_grouped_rf_summary = pd.DataFrame({
    "model": ["RandomForest_fixed_DOI_grouped"],
    "n_splits": [len(DOI_GROUPED_RF_RESULTS)],
    "PCE_R2_mean": [DOI_GROUPED_RF_RESULTS["PCE_R2"].mean()],
    "PCE_R2_std": [DOI_GROUPED_RF_RESULTS["PCE_R2"].std()],
    "PCE_R2_min": [DOI_GROUPED_RF_RESULTS["PCE_R2"].min()],
    "PCE_R2_max": [DOI_GROUPED_RF_RESULTS["PCE_R2"].max()],
    "PCE_RMSE_mean": [DOI_GROUPED_RF_RESULTS["PCE_RMSE"].mean()],
    "PCE_RMSE_std": [DOI_GROUPED_RF_RESULTS["PCE_RMSE"].std()],
    "n_raw_features": [DOI_GROUPED_RF_RESULTS["n_raw_features"].mean()],
    "n_encoded_features": [DOI_GROUPED_RF_RESULTS["n_encoded_features"].mean()],
    "n_estimators": [N_ESTIMATORS],
    "min_samples_leaf": [MIN_SAMPLES_LEAF],
    "rf_random_state": [RF_RANDOM_STATE],
    "mean_test_doi_groups": [DOI_GROUPED_RF_RESULTS["n_test_doi"].mean()],
    "mean_overlapping_doi": [DOI_GROUPED_RF_RESULTS["n_overlapping_doi"].mean()],
})

display(doi_grouped_rf_summary)


,model,n_splits,PCE_R2_mean,PCE_R2_std,PCE_R2_min,PCE_R2_max,PCE_RMSE_mean,PCE_RMSE_std,n_raw_features,n_encoded_features,n_estimators,min_samples_leaf,rf_random_state,mean_test_doi_groups,mean_overlapping_doi
0,RandomForest_fixed_DOI_grouped,10,0.300749,0.07376,0.136649,0.402252,3.931573,0.137838,207.0,3824.5,200,1,42,165.0,0.0


### Source-aware validation interpretation

- Repeated random held-out splits estimate prediction performance when train
  and test are drawn from the same mixed literature distribution.
- DOI-grouped splits estimate performance on unseen papers/sources, which is a
  stricter domain-shift setting.
- The canonical 207 features, preprocessing, fixed Random Forest, test fraction,
  and ten seeds are held constant. The intended experimental difference is only
  random row-wise splitting versus DOI-grouped splitting.
- Because DOI groups are small and highly fragmented, DOI-level validation is
  used as a diagnostic stress test rather than as the sole headline evaluation.
- The exact numerical conclusion must be updated from this fixed-RF,
  207-feature run; neither the obsolete selected-50 result nor the tuned-RF
  result should be reused.
